# numpy + scipy — Equity curve analytics & drawdown attribution

## Project goal

Compute equity curves and drawdowns for each crypto asset, characterise the return distribution (moments + normality test), and bootstrap a 95% CI on annualised Sharpe per asset. Produce one summary table comparing all four assets on (Sharpe, CI, max drawdown, drawdown duration, kurtosis).


## Why this exercises the cheatsheet

Equity-curve and drawdown analysis is *fundamentally* numpy work — cumulative products, running maxima, vectorised broadcasting. scipy.stats covers the distributional characterisation (skew/kurtosis/Jarque-Bera). Forces you to stay vectorised; any `for` loop here is a code smell.


## Sub-task 1: Build equity curves from log returns

Load the parquet, compute hourly log returns per symbol, then build a per-symbol equity curve assuming start-equity = 1.0. The equity at time t is `exp(cumsum(log_returns up to t))`. Plot all four curves on one chart for sanity (you can use matplotlib for this step).

**Patterns this forces:**

- `np.log(close).diff()` (or pandas equivalent — wrap in `np.asarray` for the math steps)
- `np.cumsum` to get cumulative log returns
- `np.exp` to convert to equity multiplier
- broadcasting across the (n_steps, n_symbols) array


In [ ]:
# Your answer here

## Sub-task 2: Drawdowns via running max

For each asset, compute the drawdown series: `(equity - running_max) / running_max`. Use `np.maximum.accumulate(equity)` (vectorised). Report max drawdown per asset and the timestamp at which it occurred.

**Patterns this forces:**

- `np.maximum.accumulate` (NOT a `for` loop)
- `np.argmin` to locate the worst drawdown
- broadcasting `(equity - running_max) / running_max`


In [ ]:
# Your answer here

## Sub-task 3: Drawdown duration

For each asset, find the longest 'underwater' stretch: the run of consecutive bars where drawdown < 0. Report the start, end, and duration in hours. Hint: a boolean mask + `np.diff` on the run-length encoded transitions.

**Patterns this forces:**

- boolean mask + `np.diff` for run-length encoding
- `np.where` to find transition indices
- vectorised search across the array


In [ ]:
# Your answer here

## Sub-task 4: Distributional moments

For each asset's hourly log returns, compute skew, excess kurtosis (Fisher), and the Jarque-Bera test for normality. Produce a small DataFrame with columns `(asset, skew, excess_kurt, jb_stat, jb_pvalue, is_normal_at_5pct)`.

**Patterns this forces:**

- `scipy.stats.skew`, `scipy.stats.kurtosis(fisher=True)`
- `scipy.stats.jarque_bera`
- vectorised iteration over assets (dict comprehension or apply across columns)


In [ ]:
# Your answer here

## Sub-task 5: Bootstrap annualised Sharpe with 95% CI

For each asset, bootstrap the annualised Sharpe ratio: resample hourly returns with replacement, compute Sharpe = `mean / std * sqrt(24*365)`, repeat 1000×, return [2.5%, 97.5%] percentiles. Use `np.random.default_rng` and integer indices, NOT a pandas-row resample.

**Patterns this forces:**

- `rng.integers(0, n, n)` for index resampling
- vectorised mean / std on each resample
- `np.quantile(boot_samples, [0.025, 0.975])`


In [ ]:
# Your answer here

## Sub-task 6: Two-sample distribution test (KS + Welch)

Test whether BTC and ETH hourly returns come from the same distribution (KS test) AND have the same mean (Welch's t-test). Report p-values and interpret in plain English: do they look statistically different on either dimension?

**Patterns this forces:**

- `scipy.stats.ks_2samp`
- `scipy.stats.ttest_ind(equal_var=False)`
- interpretation in plain English in a comment


In [ ]:
# Your answer here

## Sub-task 7: Final summary table

Combine outputs from sub-tasks 2, 4, 5 into one DataFrame: per asset, (max_drawdown, max_dd_duration_hours, skew, kurtosis, sharpe_lo, sharpe_hi, jb_pvalue). This is the deliverable.

**Patterns this forces:**

- `pd.DataFrame(rows)` from a list of dicts
- consistent column ordering
- rounding for readability via `.round(...)`


In [ ]:
# Your answer here

## What success looks like

- One summary table with one row per asset, ~7 columns.
- All numeric work is vectorised: zero `for` loops over rows.
- Bootstrap completes in <2 seconds per asset (uses array indexing, not iteration).
- You can answer at a glance: which asset has the best risk-adjusted return, and is the difference statistically significant?
